# 📰 NewsScope: News Article Category Classifier
### Master of Computer Applications (MCA) Project · School of Computer and System Science, JNU
* **Author:** Abhijeetkumar Gautam (Reg No: `25/10/JC/057`)
* **Project Guide:** Dr. Vikas Mittal
* **Academic Year:** 2026–2027

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet0848/NewsLens/blob/main/notebooks/NewsScope_Classifier.ipynb)

---
### 📌 Project Overview
This Google Colab notebook implements an end-to-end Machine Learning NLP pipeline that classifies news articles into **5 BBC domains**:
1. **Business**
2. **Entertainment**
3. **Politics**
4. **Sport**
5. **Technology**

We train, benchmark, and compare **two distinct machine learning paradigms** over the dataset:
* 🎲 **1. Multinomial Naive Bayes (MNB)** — *Probabilistic Generative Baseline (Multiplies word likelihoods via Bayes' Theorem)*
* ⚡ **2. Linear Support Vector Machine (Linear SVM)** — *Maximum-Margin Discriminative Winner ⭐ (Draws optimal separating boundary)*


In [ ]:
# ==============================================================================
# SECTION 1: Environment Setup & Library Imports
# ==============================================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn nltk joblib

import os
import re
import time
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Toolkit
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning & Evaluation
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
import joblib

# Set aesthetic visual theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
print("✅ All required libraries imported successfully!")

---
## 2. Dataset Acquisition & Validation
We use the **BBC News Dataset** (Greene & Cunningham, ICML 2006), containing 2,225 authentic news stories.
The cell below automatically downloads the dataset directly from the project repository if running inside Google Colab.

In [ ]:
# ==============================================================================
# SECTION 2: Load BBC News Corpus
# ==============================================================================
DATASET_FILENAME = "bbc-news-data.csv"
DATASET_URL = "https://raw.githubusercontent.com/Abhijeet0848/NewsLens/main/data/bbc-news-data.csv"

# Check local paths or download from GitHub repository
if not os.path.exists(DATASET_FILENAME):
    if os.path.exists(os.path.join("..", "data", DATASET_FILENAME)):
        DATASET_FILENAME = os.path.join("..", "data", DATASET_FILENAME)
    elif os.path.exists(os.path.join("data", DATASET_FILENAME)):
        DATASET_FILENAME = os.path.join("data", DATASET_FILENAME)
    else:
        print(f"📥 Downloading dataset from: {DATASET_URL}")
        urllib.request.urlretrieve(DATASET_URL, DATASET_FILENAME)
        print("✅ Download complete!")

# Load into Pandas DataFrame
try:
    df = pd.read_csv(DATASET_FILENAME, sep="\t", encoding="utf-8")
    if "category" not in df.columns or "content" not in df.columns:
        df = pd.read_csv(DATASET_FILENAME, encoding="utf-8")
except Exception:
    df = pd.read_csv(DATASET_FILENAME, encoding="utf-8")

# Standardize columns
if "title" in df.columns and "content" in df.columns:
    df["text"] = df["title"].fillna("") + " " + df["content"].fillna("")
elif "content" in df.columns:
    df["text"] = df["content"].fillna("")

df = df.dropna(subset=["text", "category"])
df["category"] = df["category"].str.strip().str.capitalize()
df = df[df["text"].str.strip() != ""]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"📊 Dataset successfully loaded: {len(df)} unique articles across {df['category'].nunique()} categories\n")
print(df["category"].value_counts())

In [ ]:
# ==============================================================================
# Exploratory Data Analysis (EDA): Category Distribution Plot
# ==============================================================================
plt.figure(figsize=(9, 4.5), dpi=100)
colors = ["#2563eb", "#059669", "#d97706", "#dc2626", "#7c3aed"]
ax = sns.barplot(
    x=df["category"].value_counts().index,
    y=df["category"].value_counts().values,
    palette=colors
)
plt.title("BBC News Dataset: Article Distribution across 5 Domains", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("News Domain", fontsize=11, fontweight="bold")
plt.ylabel("Number of Articles", fontsize=11, fontweight="bold")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() - 35),
                ha='center', va='center', fontsize=10, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Natural Language Processing (NLP) Preprocessing Pipeline
Raw text is cleaned in 4 sequential stages:
1. **Normalization & Cleaning:** Strip HTML tags, URLs, email addresses, and non-alphabetic symbols; lowercase all characters.
2. **Tokenization:** Split text into discrete words.
3. **Stopword Filtering:** Remove high-frequency, non-informative words (*"the", "is", "at"*).
4. **Morphological Lemmatization:** Transform inflected words into their canonical dictionary root (*lemma*) (*"running"* $\rightarrow$ *"run"*, *"corporations"* $\rightarrow$ *"corporation"*).

In [ ]:
# ==============================================================================
# SECTION 3: NLP Preprocessing Function
# ==============================================================================
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

try:
    nltk_stopwords = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
except Exception:
    nltk_stopwords = {"the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "is", "was", "are", "were"}
    lemmatizer = None

def preprocess_text(text: str) -> str:
    """Cleans, tokenizes, removes stopwords, and lemmatizes raw news text."""
    if not isinstance(text, str):
        return ""
    # 1. Lowercase
    text = text.lower()
    # 2. Strip HTML, URLs, Emails
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    # 3. Strip Non-alphabetic
    text = re.sub(r"[^a-z\s]", " ", text)
    # 4. Tokenize, Stopword Filter & Lemmatize
    words = text.split()
    cleaned = []
    for w in words:
        if len(w) > 2 and w not in nltk_stopwords:
            if lemmatizer:
                try:
                    w = lemmatizer.lemmatize(w)
                except Exception:
                    pass
            cleaned.append(w)
    return " ".join(cleaned)

# Demonstrate Preprocessing on Sample Article
sample_raw = df["text"].iloc[0]
sample_clean = preprocess_text(sample_raw)
print("--- [BEFORE PREPROCESSING] ---")
print(sample_raw[:180] + "...\n")
print("--- [AFTER PREPROCESSING] ---")
print(sample_clean[:180] + "...")

# Preprocess whole corpus
print("\n🔄 Preprocessing full corpus (takes ~12-15 seconds)...")
t0 = time.time()
df["clean_text"] = df["text"].apply(preprocess_text)
print(f"✅ Preprocessing completed in {time.time() - t0:.2f} seconds!")

---
## 4. Stratified Dataset Splitting & TF-IDF Vectorization
To ensure **zero data leakage**, the feature space is configured as follows:
* **Stratified 80/20 Split:** Preserves equal category proportions across train (80%) and test (20%) partitions.
* **TF-IDF Feature Extraction:** Strictly fitted **only** on the training data (`max_features=5000`, `ngram_range=(1, 2)` unigrams + bigrams, sublinear TF scaling).

In [ ]:
# ==============================================================================
# SECTION 4: Train/Test Split & TF-IDF Feature Extraction
# ==============================================================================
classes = sorted(df["category"].unique().tolist())
le = LabelEncoder()
le.fit(classes)
y_all = le.transform(df["category"])

# 80/20 Stratified Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["clean_text"],
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

print(f"📊 Training set: {len(X_train_raw)} articles (80%)")
print(f"📊 Testing set:  {len(X_test_raw)} articles (20%)")

# TF-IDF Feature Extraction (Fit on Train only)
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

print(f"✅ TF-IDF Matrix Shape: {X_train_tfidf.shape} ({X_train_tfidf.nnz:,} non-zero entries)")

---
## 5. Training the 2 Core Classifiers: Naive Bayes vs. Linear SVM
We train and benchmark the **two foundational NLP classification algorithms**:
1. 🎲 **Multinomial Naive Bayes (MNB):** Generative probabilistic model applying Bayes' Theorem with Laplace smoothing ($lpha = 0.1$).
2. ⚡ **Linear SVM (Support Vector Machine):** Maximum-margin hyperplane with Platt scaling calibration ($C = 1.0$, 5-fold CV).


In [ ]:
# ==============================================================================
# SECTION 5: Train Both Models & Measure Inference Latency
# ==============================================================================
models = {
    "naive_bayes": {
        "name": "Multinomial Naive Bayes",
        "paradigm": "Generative Probabilistic",
        "model": MultinomialNB(alpha=0.1),
    },
    "linear_svm": {
        "name": "Linear Support Vector Machine",
        "paradigm": "Maximum-Margin Hyperplane",
        "model": CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000), cv=5),
    },
}

results = {}
trained_models = {}

print("=" * 85)
print(f"{'Model':<35} | {'Accuracy':<9} | {'Macro F1':<9} | {'Latency/Doc':<12} | {'Paradigm'}")
print("=" * 85)

for key, spec in models.items():
    clf = spec["model"]

    # Measure Training Time
    t0 = time.time()
    clf.fit(X_train_tfidf, y_train)
    train_time = time.time() - t0

    # Measure Inference Latency
    t_pred = time.time()
    y_pred = clf.predict(X_test_tfidf)
    latency_ms = ((time.time() - t_pred) / len(y_test)) * 1000.0

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    cm = confusion_matrix(y_test, y_pred)

    results[key] = {
        "name": spec["name"],
        "paradigm": spec["paradigm"],
        "accuracy": acc,
        "f1": f1,
        "latency_ms": latency_ms,
        "train_time": train_time,
        "confusion_matrix": cm,
        "y_pred": y_pred,
    }
    trained_models[key] = clf

    print(f"{spec['name']:<35} | {acc*100:>7.2f}% | {f1*100:>7.2f}% | {latency_ms:>9.3f} ms | {spec['paradigm']}")

print("=" * 85)
winner = max(results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Performing Model: {winner[1]['name']} with {winner[1]['accuracy']*100:.2f}% Accuracy!")


---
## 6. Model Evaluation & Visual Benchmarks
Let's visualize the performance comparison between **Multinomial Naive Bayes** and **Linear SVM** with **Accuracy/F1 comparison charts** and **Confusion Matrix heatmaps**.


In [ ]:
# ==============================================================================
# SECTION 6.1: Accuracy & Macro F1 Comparison Bar Chart
# ==============================================================================
model_names = [res["name"] for res in results.values()]
accuracies = [res["accuracy"] * 100 for res in results.values()]
f1_scores = [res["f1"] * 100 for res in results.values()]

x = np.arange(len(model_names))
width = 0.30

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=100)
rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy (%)', color='#2563eb', edgecolor='black', alpha=0.9)
rects2 = ax.bar(x + width/2, f1_scores, width, label='Macro F1 (%)', color='#059669', edgecolor='black', alpha=0.9)

ax.set_ylabel('Score (%)', fontweight='bold', fontsize=11)
ax.set_title('Comparative Performance: Multinomial Naive Bayes vs. Linear SVM', fontweight='bold', fontsize=13, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontweight='bold', fontsize=11)
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
ax.set_ylim(90, 100)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Add value labels on top of bars
for rect in rects1:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=10, color='#1e40af')

for rect in rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=10, color='#047857')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# SECTION 6.2: Side-by-Side Confusion Matrices (Naive Bayes vs. Linear SVM)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=100)
cmaps = ["Blues", "Greens"]

for idx, (key, res) in enumerate(results.items()):
    ax = axes[idx]
    sns.heatmap(
        res["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap=cmaps[idx],
        xticklabels=classes,
        yticklabels=classes,
        cbar=True,
        ax=ax,
        linewidths=0.5
    )
    ax.set_title(f"{res['name']}\nAccuracy: {res['accuracy']*100:.2f}% | F1: {res['f1']*100:.2f}%", fontweight='bold', fontsize=11, pad=10)
    ax.set_xlabel('Predicted Category', fontweight='bold')
    ax.set_ylabel('True Category', fontweight='bold')

plt.suptitle('Confusion Matrix Comparison: BBC News 5-Class Categorization', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# SECTION 6.3: Detailed Classification Reports for Both Models
# ==============================================================================
for key, res in results.items():
    print(f"\n{'='*70}")
    print(f"📋 CLASSIFICATION REPORT: {res['name'].upper()} ({res['paradigm']})")
    print(f"{'='*70}")
    print(classification_report(y_test, res["y_pred"], target_names=classes, digits=4))


---
## 7. Interactive Live Prediction & Explainable AI (XAI)
Test the trained models with custom news articles! The function below:
1. Preprocesses the raw text.
2. Generates calibrated probabilities across all 5 news domains.
3. Extracts the **top salient keywords** driving the classification decision.

In [ ]:
# ==============================================================================
# SECTION 7: Live Prediction & Keyword Saliency Function
# ==============================================================================
feature_names = vectorizer.get_feature_names_out()

def classify_article(article_text: str, model_key: str = "linear_svm"):
    """Classifies an article, prints probabilities, and extracts salient keywords."""
    clean = preprocess_text(article_text)
    X_vec = vectorizer.transform([clean])

    clf = trained_models[model_key]
    model_name = results[model_key]["name"]

    # Calculate probabilities
    probs = clf.predict_proba(X_vec)[0]
    pred_idx = int(np.argmax(probs))
    pred_class = classes[pred_idx]
    confidence = probs[pred_idx] * 100

    # Keyword Saliency (TF-IDF weights)
    cx = X_vec.tocoo()
    sorted_keywords = sorted(
        [{"word": str(feature_names[c]), "weight": round(float(v), 4)} for c, v in zip(cx.col, cx.data)],
        key=lambda x: x["weight"],
        reverse=True
    )
    top_keywords = sorted_keywords[:6]

    print("=" * 70)
    print(f"📰 INPUT: \"{article_text[:90]}...\"")
    print(f"🤖 MODEL: {model_name}")
    print(f"🎯 PREDICTION: {pred_class.upper()} ({confidence:.1f}% Calibrated Confidence)")
    print("-" * 70)
    print("📊 Probability Distribution:")
    for c_name, p in zip(classes, probs):
        bar = "█" * int(p * 25)
        print(f"  {c_name:<15}: {p*100:>5.1f}%  {bar}")
    print("-" * 70)
    print("🔍 Top Discriminative Saliency Keywords:")
    for kw in top_keywords:
        print(f"  • {kw['word']:<18} (TF-IDF Weight: +{kw['weight']:.4f})")
    print("=" * 70 + "\n")

# Test with 3 Real-World Articles across different categories
classify_article(
    "Apple announces new M3 MacBook laptop with ultra-fast silicon processor and 22-hour battery life for software developers.",
    model_key="linear_svm"
)

classify_article(
    "Real Madrid secures Champions League title after thrilling penalty shootout in the final football match.",
    model_key="linear_svm"
)

classify_article(
    "Central Bank raises benchmark interest rates by 50 basis points as inflation growth impacts corporate earnings and bond markets.",
    model_key="linear_svm"
)

---
## 8. Exporting Model Artifacts for Web Deployment
Serializes the trained models into lightweight joblib/pickle files that can be loaded into the **NewsScope Next.js / FastAPI** web dashboard.

In [ ]:
# ==============================================================================
# SECTION 8: Model Serialization
# ==============================================================================
os.makedirs("exported_models", exist_ok=True)

joblib.dump(vectorizer, "exported_models/vectorizer.pkl", compress=3)
joblib.dump(le, "exported_models/label_encoder.pkl")
joblib.dump(trained_models["linear_svm"], "exported_models/svm.pkl", compress=3)
joblib.dump(trained_models["naive_bayes"], "exported_models/nb.pkl", compress=3)

print("✅ Both models and vectorizers successfully exported to './exported_models/' directory!")
print("   • svm.pkl (Linear SVM Champion)")
print("   • nb.pkl (Multinomial Naive Bayes)")
print("   • vectorizer.pkl (Fitted TF-IDF Vectorizer)")
print("   • label_encoder.pkl (5-Class Label Encoder)")
